# SmolLM2 safetensors on PYNQ-Z1
This notebook reads the original full 30-layer checkpoint using the unchanged `src` safetensors loader, then runs the AOT package compiled from that checkpoint on the physical NPU. Compilation and W8A8 quantization happen on the host; the board does not compile the checkpoint. Run all cells in the root PYNQ Python kernel. Change the prompt in the generation cell.

In [ ]:
import os, sys, json, hashlib, platform
from pathlib import Path
import numpy as np
ROOT = Path(os.environ.get("NPU_SMOLLM_DEMO", ".")).resolve()
sys.path.insert(0, str(ROOT))
assert platform.machine() == "armv7l", "Run this notebook on PYNQ-Z1"
assert os.geteuid() == 0, "Use the root PYNQ kernel for MMIO and CMA"
def digest(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()
seal = json.loads((ROOT / "deployment.json").read_text())
for name, expected in seal["files"].items():
    assert digest(ROOT / name) == expected, "Deployment differs: " + name
print("Verified deployment:", ROOT)

## Load the original safetensors checkpoint
The loader memory maps the checkpoint. Read and validate actual tensor values without allocating a second full model on this 512 MB board. The compiled package contains the quantized weights used for inference.

In [ ]:
from src.export.core.model_loader import SafeTensors
spec = json.loads((ROOT / "model-source.json").read_text())
checkpoint = ROOT / "checkpoint"
for name, info in spec["files"].items():
    assert digest(checkpoint / name) == info["sha256"], name
config = json.loads((checkpoint / "config.json").read_text())
assert config["num_hidden_layers"] == 30
with SafeTensors(checkpoint) as tensors:
    names = tensors.keys()
    assert "model.layers.29.mlp.down_proj.weight" in names
    norm = tensors["model.norm.weight"]
    assert norm.shape == (config["hidden_size"],) and np.isfinite(norm).all()
print("Loaded safetensors:", len(names), "tensors;", config["num_hidden_layers"], "layers")

## Load the compiled model and physical overlay
Only the repository runtime and vendor driver execute the model. Full W8 weights are streamed through CMA; KV cache uses FP32. No model layers are truncated.

The default prompt is checked against an independent ISA simulation of the same ARM library. The x86 host result is recorded separately because floating-point execution can change greedy token choices across CPU targets.

In [ ]:
from pynq import Overlay
from src.runtime.core.engine import LLMEngine
package = ROOT / "package"
manifest = json.loads((package / "manifest.json").read_text())
assert manifest["meta"]["config"]["layers"] == config["num_hidden_layers"]
assert manifest["meta"]["family"] == "llama"
assert manifest["npu"]["tasks"] > 0
overlay = Overlay(str(ROOT / "npu_matrix.bit"))
engine = LLMEngine(package, backend="pynq", overlay=overlay, stream_weights=True)
print("Physical NPU ready; tasks:", manifest["npu"]["tasks"])

In [ ]:
prompt = "Write a paragraph about a neural processing unit (NPU)."
max_new_tokens = 32
ids, text, stats = engine.generate(
    prompt, max_new_tokens=max_new_tokens, chat=True,
    on_token=lambda _, piece: print(piece, end="", flush=True),
)
print()
runtime = engine.package.stats()
assert len(ids) > 1 and text.strip(), "No generated response"
assert runtime["calls"] > 0 and runtime["jobs"] > 0 and runtime["error"] == 0
reference = json.loads((ROOT / "arm-reference.json").read_text())
reference_checked = prompt == reference["prompt"] and max_new_tokens == reference["max_new_tokens"]
reference_match = not reference_checked or ids == reference["output_ids"]
result = {"success": reference_match, "reference_checked": reference_checked,
          "reference_match": reference_match, "prompt": prompt, "output_ids": ids, "output": text,
          "stats": stats.as_dict(), "runtime": runtime,
          "checkpoint_sha256": spec["files"]["model.safetensors"]["sha256"],
          "layers": config["num_hidden_layers"], "backend": "pynq"}
(ROOT / "result.json").write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))
assert reference_match, "Physical NPU output differs from the ARM ISA reference; see result.json and docs/board-run/README.md"

## Release board buffers
Run after generation, or before creating a new engine.

In [ ]:
for buffer in (engine.package.io, engine.package.stage, engine.package.words):
    if buffer is not None:
        buffer.buffer.freebuffer()
del engine
print("Board buffers released")